<a href="https://colab.research.google.com/github/VitorKF/Aulas-Programacao-para-Ciencia-de-Dados/blob/main/Estudo_Independente_III_%E2%80%93_Coleta_de_Dados_utilizando_APIs_ou_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import requests
import pandas as pd
import time

BASE_URL = "https://swapi.dev/api/"

def coletar_dados(categoria):
    url = f"{BASE_URL}{categoria}/"
    registros = []

    while url:
        resposta = requests.get(url, timeout=30)
        resposta.raise_for_status()

        dados = resposta.json()
        registros.extend(dados["results"])

        url = dados["next"]
        time.sleep(0.5)

    print(f"{categoria.capitalize()}: {len(registros)} registros")

    return registros

In [8]:
personagens = coletar_dados("people")
planetas = coletar_dados("planets")
filmes = coletar_dados("films")
especies = coletar_dados("species")
veiculos = coletar_dados("vehicles")
naves = coletar_dados("starships")

print("\nColeta geral finalizada!")
print("Total de registros coletados:",
      sum(map(len, [
          personagens, planetas, filmes,
          especies, veiculos, naves
      ])))

People: 82 registros
Planets: 60 registros
Films: 6 registros
Species: 37 registros
Vehicles: 39 registros
Starships: 36 registros

Coleta geral finalizada!
Total de registros coletados: 260


In [9]:
# Criar mapas de relacionamento: URL -> Nome
mapas = {
    "people": {
        item["url"]: item["name"] for item in personagens
    },
    "planets": {
        item["url"]: item["name"] for item in planetas
    },
    "films": {
        item["url"]: item["title"] for item in filmes
    },
    "species": {
        item["url"]: item["name"] for item in especies
    },
    "vehicles": {
        item["url"]: item["name"] for item in veiculos
    },
    "starships": {
        item["url"]: item["name"] for item in naves
    }
}

# Função para substituir URLs pelos nomes
def resolver_referencia(valor, categoria):
    mapa = mapas[categoria]

    if isinstance(valor, list):
        return "; ".join(
            mapa.get(url, url) for url in valor
        )

    if isinstance(valor, str):
        return mapa.get(valor, valor)

    return valor

In [10]:

# Personagens
personagens_tratados = []

for personagem in personagens:
    item = personagem.copy()

    item["homeworld"] = resolver_referencia(
        item["homeworld"], "planets"
    )
    item["films"] = resolver_referencia(
        item["films"], "films"
    )
    item["species"] = resolver_referencia(
        item["species"], "species"
    )
    item["vehicles"] = resolver_referencia(
        item["vehicles"], "vehicles"
    )
    item["starships"] = resolver_referencia(
        item["starships"], "starships"
    )

    personagens_tratados.append(item)


# Planetas
planetas_tratados = []

for planeta in planetas:
    item = planeta.copy()

    item["films"] = resolver_referencia(
        item["films"], "films"
    )
    item["residents"] = resolver_referencia(
        item["residents"], "people"
    )

    planetas_tratados.append(item)


# Naves espaciais
naves_tratadas = []

for nave in naves:
    item = nave.copy()

    item["films"] = resolver_referencia(
        item["films"], "films"
    )
    item["pilots"] = resolver_referencia(
        item["pilots"], "people"
    )

    naves_tratadas.append(item)

print("Relacionamentos processados!")

Relacionamentos processados!


In [11]:
df_personagens = pd.DataFrame(personagens_tratados)
df_planetas = pd.DataFrame(planetas_tratados)
df_naves = pd.DataFrame(naves_tratadas)

# Selecionar colunas dos personagens
df_personagens = df_personagens[[
    "name", "height", "mass", "hair_color",
    "skin_color", "eye_color", "birth_year",
    "gender", "homeworld", "films", "species",
    "vehicles", "starships"
]].copy()

# Selecionar colunas dos planetas
df_planetas = df_planetas[[
    "name", "rotation_period", "orbital_period",
    "diameter", "climate", "gravity", "terrain",
    "surface_water", "population", "films", "residents"
]].copy()

# Selecionar colunas das naves
df_naves = df_naves[[
    "name", "model", "manufacturer", "cost_in_credits",
    "length", "max_atmosphering_speed", "crew",
    "passengers", "cargo_capacity", "consumables",
    "hyperdrive_rating", "MGLT", "starship_class",
    "films", "pilots"
]].copy()

# Converter campos numéricos
colunas_personagens = ["height", "mass"]

colunas_planetas = [
    "rotation_period", "orbital_period",
    "diameter", "surface_water", "population"
]

colunas_naves = [
    "cost_in_credits", "length",
    "max_atmosphering_speed", "crew",
    "passengers", "cargo_capacity",
    "hyperdrive_rating", "MGLT"
]

for coluna in colunas_personagens:
    df_personagens[coluna] = pd.to_numeric(
        df_personagens[coluna], errors="coerce"
    )

for coluna in colunas_planetas:
    df_planetas[coluna] = pd.to_numeric(
        df_planetas[coluna], errors="coerce"
    )

for coluna in colunas_naves:
    df_naves[coluna] = pd.to_numeric(
        df_naves[coluna], errors="coerce"
    )

print("DataFrames preparados!")

DataFrames preparados!


In [12]:
df_personagens.to_csv(
    "star_wars_personagens.csv",
    index=False,
    encoding="utf-8-sig"
)

df_planetas.to_csv(
    "star_wars_planetas.csv",
    index=False,
    encoding="utf-8-sig"
)

df_naves.to_csv(
    "star_wars_naves.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos CSV exportados com sucesso!")

Arquivos CSV exportados com sucesso!


In [13]:
from google.colab import files

files.download("star_wars_personagens.csv")
files.download("star_wars_planetas.csv")
files.download("star_wars_naves.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>